# 09 — Panel Assembly: integração final do pipeline ETL

**Última camada do pipeline.** Junta os 7 outputs de `data/interim/` em 3 painéis balanceados prontos para modelagem (CS, SDID, TWFE).

**Decisões metodológicas (§3.4 v2.2):**
- A1: 3 painéis (full 2012-2024, main 2015-2024, canavieiro_main 2015-2024).
- A2: 15 colunas estratégicas do MapBiomas (das 51 originais).
- A3: `g_m = NaN` para não-tratados (convenção CS/Synth).

**Inputs (todos em `data/interim/`):**
1. `crosswalk_centrosul.csv` — 2.363 munis (universo)
2. `seeg_outcomes_audited.csv` — 5 outcomes AFOLU + transformações
3. `sicar_outcomes_anual.csv` — 3 outcomes H1c
4. `anp_muni_treat.csv` — tratamento (g_m, doses)
5. `pam_cana_wide.csv` — controles dinâmicos
6. `mapbiomas_panel.csv` — uso do solo (15 cols selecionadas)
7. `psm_baseline_clean.csv` — covariáveis pré-tratamento (71 cols)
8. `universo_canavieiro_final.csv` — flag canavieiro

**Outputs em `data/interim/`:**
- `panel_full_2012_2024.csv` — 2.363 × 13 = 30.719 cells × ~135 cols
- `panel_main_2015_2024.csv` — 2.363 × 10 = 23.630 cells × ~135 cols
- `panel_canavieiro_main.csv` — 842 × 10 = 8.420 cells × ~135 cols

**Tempo esperado:** ~30 segundos.

In [1]:
# Setup portável — resolve a raiz do repositório sem depender do Google Drive.
# Para executar a partir do Drive, defina antes: os.environ["RENOVABIO_BASE_DIR"] = "<caminho>"
# Para executar a partir do Drive, defina antes de rodar esta célula:
import os
import sys
from pathlib import Path

if os.environ.get("RENOVABIO_BASE_DIR"):
    BASE_DIR = Path(os.environ["RENOVABIO_BASE_DIR"]).expanduser().resolve()
else:
    BASE_DIR = Path.cwd().resolve()
    while not (BASE_DIR / "requirements.txt").exists() and BASE_DIR != BASE_DIR.parent:
        BASE_DIR = BASE_DIR.parent

if str(BASE_DIR) not in sys.path:
    sys.path.insert(0, str(BASE_DIR))
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

Mounted at /content/drive


In [2]:
# Reload módulos
import importlib
from pipeline import config, normalize, panel_assembly
importlib.reload(config); importlib.reload(normalize); importlib.reload(panel_assembly)

from pipeline.config import PARAMS, interim, out_pre
from pipeline.panel_assembly import (
    run_panel_assembly, panel_summary,
    ANOS_MAIN, ANOS_FULL, MAPBIOMAS_COLS_KEEP,
)
print('✓ módulos carregados')
print(f'  ANOS_FULL: {ANOS_FULL[0]}-{ANOS_FULL[-1]} ({len(ANOS_FULL)} anos)')
print(f'  ANOS_MAIN: {ANOS_MAIN[0]}-{ANOS_MAIN[-1]} ({len(ANOS_MAIN)} anos)')
print(f'  MAPBIOMAS_COLS_KEEP: {len(MAPBIOMAS_COLS_KEEP)}')

✓ módulos carregados
  ANOS_FULL: 2012-2024 (13 anos)
  ANOS_MAIN: 2015-2024 (10 anos)
  MAPBIOMAS_COLS_KEEP: 15


## Roda assembly final

In [3]:
# Verifica que todos os 7 prerequisitos existem
required = [
    'crosswalk_centrosul.csv',
    'seeg_outcomes_audited.csv',
    'sicar_outcomes_anual.csv',
    'anp_muni_treat.csv',
    'pam_cana_wide.csv',
    'mapbiomas_panel.csv',
    'psm_baseline_clean.csv',
    'universo_canavieiro_final.csv',
]
missing = [f for f in required if not interim(f).exists()]
if missing:
    raise FileNotFoundError(
        f'⚠️ Pré-requisitos faltando em data/interim/:\n  ' +
        '\n  '.join(missing) + '\n\nRode os notebooks 01-07 antes deste.'
    )
print('✓ todos os 8 pré-requisitos presentes\n')

result = run_panel_assembly(save=True)

✓ todos os 8 pré-requisitos presentes

→ Carregando 7 inputs de data/interim/...
  crosswalk: (2363, 5)
  seeg: (23630, 19)
  sicar: (10191, 18)
  anp: (194, 13)
  pam: (22837, 8)
  mapbiomas: (30719, 56)
  psm: (2363, 71)
  universo: (842, 16)

→ Construindo painéis...

  Janela: 2012-2024 (13 anos)
    grid base: (30719, 4)
    panel completo: (30719, 139)

  Janela: 2015-2024 (10 anos)
    grid base: (23630, 4)
    panel completo: (23630, 139)

  panel_canavieiro_main (subset main): (8420, 139)
    842 munis × 10 anos = 8420 cells esperadas

→ Salvando 3 painéis...
  ✓ tudo salvo


## Sumário dos 3 painéis

In [4]:
for label, df in [('panel_full_2012_2024', result['panel_full']),
                   ('panel_main_2015_2024', result['panel_main']),
                   ('panel_canavieiro_main', result['panel_canavieiro'])]:
    s = panel_summary(df, label)
    print(f'\n=== {label} ===')
    for k, v in s.items():
        if k != 'label':
            print(f'  {k:25s}: {v}')


=== panel_full_2012_2024 ===
  shape                    : (30719, 139)
  n_munis                  : 2363
  n_anos                   : 13
  anos_range               : (2012, 2024)
  n_tratados_ever          : 194
  n_canavieiros            : 842
  n_cols                   : 139
  cov_luc                  : 23630
  cov_carbono_solo         : 23630
  cov_solos_manejados      : 23630
  cov_queima               : 23630
  cov_cobertura_car_ativo  : 10191
  cov_adesao_pra           : 10191

=== panel_main_2015_2024 ===
  shape                    : (23630, 139)
  n_munis                  : 2363
  n_anos                   : 10
  anos_range               : (2015, 2024)
  n_tratados_ever          : 194
  n_canavieiros            : 842
  n_cols                   : 139
  cov_luc                  : 23630
  cov_carbono_solo         : 23630
  cov_solos_manejados      : 23630
  cov_queima               : 23630
  cov_cobertura_car_ativo  : 9759
  cov_adesao_pra           : 9759

=== panel_canavieiro_ma

In [5]:
# Lista todas as colunas do panel_main para auditoria
panel_main = result['panel_main']
print(f'panel_main: {panel_main.shape}')
print(f'\n{len(panel_main.columns)} colunas, agrupadas por categoria:\n')

categories = {
    'identificadores': ['geocode', 'municipio', 'uf', 'ano'],
    'SEEG outcomes raw': ['luc', 'carbono_solo', 'queima', 'solos_manejados', 'residuos_florestais'],
    'SEEG transformações': [c for c in panel_main.columns if c.startswith(('asinh_', 'log1p_', 'log_solos'))],
    'SEEG flags F3': [c for c in panel_main.columns if c.startswith('flag_')],
    'SICAR (H1c)': ['cobertura_car_ativo', 'cobertura_car_ativo_pendente', 'adesao_pra',
                    'share_pra_nao_informado', 'share_veg_nativa_atual', 'is_partial_year'],
    'tratamento ANP': ['g_m', 'g_data_m', 'n_usinas', 'n_usinas_baseline'] +
                       [c for c in panel_main.columns if c.startswith('dose_T')],
    'derivações tratamento': ['is_treated_ever', 'is_treated_main',
                              'treatment_dummy_t', 'period_relative_g',
                              'post_2018', 'post_2020'] +
                              [c for c in panel_main.columns if c.startswith(('T2_dummy', 'T3_dummy'))],
    'PAM dinâmico': [c for c in panel_main.columns if c.endswith('_pam')],
    'MapBiomas dinâmico': [c for c in panel_main.columns if c.startswith('mb_area_') or c.startswith('mb_share_')],
    'flag canavieiro': ['is_canavieiro_mb', 'is_canavieiro_pam', 'is_canavieiro_anp',
                        'is_canavieiro_uniao', 'n_criterios_atendidos'],
    'PSM baseline (cross-section)': [c for c in panel_main.columns
                                      if c not in [item for sublist in {} for item in sublist]
                                      and c not in ('geocode','municipio','uf','ano')
                                      and not c.startswith(('luc','carbono_solo','queima','solos_manejados','residuos_florestais',
                                                             'asinh_','log1p_','log_solos','flag_',
                                                             'cobertura_','adesao_','share_pra','share_veg','is_partial',
                                                             'g_m','g_data','n_usinas','dose_',
                                                             'is_treated','treatment_','period_','post_','T2_dummy','T3_dummy',
                                                             'mb_','is_canavieiro','n_criterios'))
                                      and not c.endswith('_pam')]
}

for cat, cols in categories.items():
    cols_present = [c for c in cols if c in panel_main.columns]
    print(f'  [{cat}] ({len(cols_present)} cols)')
    for c in cols_present[:5]:
        print(f'      {c}')
    if len(cols_present) > 5:
        print(f'      ... +{len(cols_present)-5}')
    print()

panel_main: (23630, 139)

139 colunas, agrupadas por categoria:

  [identificadores] (4 cols)
      geocode
      municipio
      uf
      ano

  [SEEG outcomes raw] (5 cols)
      luc
      carbono_solo
      queima
      solos_manejados
      residuos_florestais

  [SEEG transformações] (5 cols)
      asinh_luc
      asinh_carbono_solo
      log1p_queima
      log_solos_manejados
      log1p_residuos_florestais

  [SEEG flags F3] (5 cols)
      flag_carbono_solo
      flag_luc
      flag_queima
      flag_residuos_florestais
      flag_solos_manejados

  [SICAR (H1c)] (6 cols)
      cobertura_car_ativo
      cobertura_car_ativo_pendente
      adesao_pra
      share_pra_nao_informado
      share_veg_nativa_atual
      ... +1

  [tratamento ANP] (10 cols)
      g_m
      g_data_m
      n_usinas
      n_usinas_baseline
      dose_T2_2022
      ... +5

  [derivações tratamento] (12 cols)
      is_treated_ever
      is_treated_main
      treatment_dummy_t
      period_relative_g
      pos

## Sample real: Quirinópolis-GO 2020

In [6]:
# Quirinópolis-GO é um dos canavieiros do top 5 — bom para audit
panel_can = result['panel_canavieiro']
quir = panel_can[(panel_can['geocode']=='5218508') & (panel_can['ano']==2020)]
if len(quir) > 0:
    s = quir.iloc[0]
    show = ['municipio','uf','ano',
            'luc','carbono_solo','queima','asinh_luc','log1p_queima',
            'cobertura_car_ativo','adesao_pra',
            'is_canavieiro_uniao','n_criterios_atendidos','is_treated_ever',
            'g_m','treatment_dummy_t','period_relative_g',
            'dose_T2_2026','dose_T3_2026',
            'pib_percap','idhm_renda','share_vab_agro','bioma',
            'mb_share_sugarcane','mb_area_sugarcane_ha',
            'area_colhida_ha_pam','post_2018','post_2020']
    for c in show:
        if c in s.index:
            v = s[c]
            if isinstance(v, float) and pd.notna(v):
                print(f'  {c:30s}: {v:.4f}')
            elif isinstance(v, bool):
                print(f'  {c:30s}: {v}')
            else:
                print(f'  {c:30s}: {v!r}')

  municipio                     : 'Quirinópolis'
  uf                            : 'GO'
  ano                           : np.int64(2020)
  luc                           : 155114.2456
  carbono_solo                  : 2575.3247
  queima                        : 634.7804
  asinh_luc                     : 12.6451
  log1p_queima                  : 6.4549
  cobertura_car_ativo           : np.float64(nan)
  adesao_pra                    : np.float64(nan)
  is_canavieiro_uniao           : np.True_
  n_criterios_atendidos         : np.int64(3)
  is_treated_ever               : np.True_
  g_m                           : 2019.0000
  treatment_dummy_t             : np.int64(1)
  period_relative_g             : 1.0000
  dose_T2_2026                  : np.float64(nan)
  dose_T3_2026                  : np.float64(nan)
  pib_percap                    : np.int64(34518)
  idhm_renda                    : 0.7320
  share_vab_agro                : 0.2266
  bioma                         : 'Cerrado'
  mb_sha

## Cobertura por outcome (panel_canavieiro_main)

In [7]:
panel_can = result['panel_canavieiro']
n_total = len(panel_can)
print(f'Total cells: {n_total:,} ({panel_can["geocode"].nunique()} munis × {panel_can["ano"].nunique()} anos)\n')
print('Cobertura por outcome:\n')
outcomes_check = [
    'luc', 'carbono_solo', 'queima', 'solos_manejados', 'residuos_florestais',
    'cobertura_car_ativo', 'adesao_pra', 'share_veg_nativa_atual',
    'mb_share_sugarcane', 'area_colhida_ha_pam',
    'pib_percap', 'idhm_renda', 'gini', 'bioma',
    'g_m', 'treatment_dummy_t',
]
for c in outcomes_check:
    if c in panel_can.columns:
        n_obs = panel_can[c].notna().sum()
        pct = 100 * n_obs / n_total
        print(f'  {c:30s}: {n_obs:>6,}/{n_total} ({pct:5.1f}%)')

Total cells: 8,420 (842 munis × 10 anos)

Cobertura por outcome:

  luc                           :  8,420/8420 (100.0%)
  carbono_solo                  :  8,420/8420 (100.0%)
  queima                        :  8,420/8420 (100.0%)
  solos_manejados               :  8,420/8420 (100.0%)
  residuos_florestais           :  8,420/8420 (100.0%)
  cobertura_car_ativo           :  3,200/8420 ( 38.0%)
  adesao_pra                    :  3,200/8420 ( 38.0%)
  share_veg_nativa_atual        :  3,200/8420 ( 38.0%)
  mb_share_sugarcane            :  8,420/8420 (100.0%)
  area_colhida_ha_pam           :  8,208/8420 ( 97.5%)
  pib_percap                    :  8,420/8420 (100.0%)
  idhm_renda                    :  8,410/8420 ( 99.9%)
  gini                          :  8,410/8420 ( 99.9%)
  bioma                         :  8,420/8420 (100.0%)
  g_m                           :  1,940/8420 ( 23.0%)
  treatment_dummy_t             :  8,420/8420 (100.0%)


In [8]:
# Distribuição da coorte g_m entre canavieiros tratados
panel_can = result['panel_canavieiro']
treated_only = panel_can[panel_can['is_treated_ever']].drop_duplicates(subset='geocode')
print(f'Canavieiros tratados (de 842): {len(treated_only)}')
print(f'\nDistribuição da coorte g_m:')
print(treated_only['g_m'].value_counts().sort_index().to_string())
print(f'\nNão-tratados (controles): {842 - len(treated_only)}')

Canavieiros tratados (de 842): 194

Distribuição da coorte g_m:
g_m
2019.0      2
2020.0    142
2021.0     40
2022.0      8
2023.0      2

Não-tratados (controles): 648


## Pipeline ETL completo ✓

Os 3 painéis estão prontos para modelagem. Próxima fase é fora do `pipeline/`:

**Próximos passos:**
- `notebook_modelagem_psm.ipynb` — propensity score matching usando `panel_canavieiro_main` (subset estrito) ou `panel_main` (todos os 2.363 munis CS)
- `notebook_modelagem_did.ipynb` — Callaway-Sant'Anna como principal, TWFE como referência, SDID como robustez
- `notebook_modelagem_dynamic.ipynb` — event-time effects via `period_relative_g`
- Atualização do **pré-registro v2.2 → v2.3** com observações empíricas:
  - §3.10: `asinh` ≡ `log1p` no CS (saldo positivo dominante)
  - §3.7.1: demover `n_usinas_baseline` para sensibilidade (apenas 2 munis baseline > 0)
  - §3.3: incluir critério MapBiomas como complementar (não-redundante com PAM)

**TODOs de auditoria pré-submissão (TODO_audit_appendix.md):**
- 7 munis MB-only (cana > 5% MB mas área < 500ha PAM)
- 24 munis PAM+ANP sem MB (PAM detecta cana mas MB share < 5%)
- Caso Vale Verde (CNPJ duplicado RN/GO)
- 2 casos `apenas_cancelado_anulado=True` no ANP